## NB01 — Data Collection

**Project**: Camino de Santiago — Contextual Drivers of Pilgrim Flow Mutations (2003–2025)  
**Phase**: Data collection — Tier 1 sources  
**Status**: ✅ Complete  

This notebook covers the full data collection pipeline for the project *"What contextual, non-demographic factors explain and predict the structural mutations of the Camino de Santiago between 2003 and 2025?"*

Four Tier 1 sources are collected and processed into clean datasets ready for exploratory analysis in NB02:

- **Google Trends** — digital visibility index for Camino-related searches across 9 countries (2004–2024)
- **Oficina del Peregrino** — official annual pilgrim statistics: totals, routes, nationalities, motives, transport modes (2004–2024)
- **Trail du Saint-Jacques by UTMB** — finisher data from 11 editions (2013–2024) as a sport-spillover proxy
- **ERA5 Climate** — monthly temperature and precipitation across 5 route corridors via Copernicus CDS (2004–2024)

All collected data is saved to `data/processed/`. To load existing data without re-running API calls, use the `# LOAD` cells in each section.

---

## How to use this notebook

All data has already been collected and saved to `data/processed/` and `data/raw/`.  
**To load existing data**: run Setup + any `# LOAD` cell. No API calls, instant.  
**To re-collect data**: run the cells marked `# COLLECTION`. Requires API access and ~30–60 min.

---

## Collection roadmap

| Section | Source | Method | Output file |
|---------|--------|--------|-------------|
| 1 | Google Trends | pytrends API | `trends_annual.csv` |
| 2 | Oficina del Peregrino | PDF extraction (2004–2018) + manual (2019–2024) | `master_annual.csv`, `master_routes.csv` |
| 3 | Trail du Saint-Jacques | LiveTrail XML scraping | `trail_sj_annual.csv`, `trail_sj_nationalities.csv` |
| 4 | ERA5 Climate | Copernicus CDS API | `era5_climate_monthly.csv` |

> **Data convention**: Raw files in `data/raw/` are never modified. All transformations produce new files in `data/processed/`.

---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
import zipfile
import shutil
import re
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
RAW       = PROJECT_ROOT / 'data' / 'raw'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL  = PROJECT_ROOT / 'data' / 'external'

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 13

print('Setup OK')
print('RAW       -> ' + str(RAW))
print('PROCESSED -> ' + str(PROCESSED))
print('EXTERNAL  -> ' + str(EXTERNAL))

---
## Section 1 — Google Trends

**Objective**: Build a digital visibility index per country and year for Camino-related searches (2004–2024).

**Approach**: One keyword per language/country, monthly series via `pytrends`, aggregated to annual means.

**Known limitation**: Google Trends returns a relative index (0–100), not absolute volumes.
Values are comparable *within* a country over time, not *across* countries.

| Country | Keyword | Language |
|---------|---------|----------|
| ES | Camino de Santiago | Spanish |
| FR | chemin de Compostelle | French |
| DE | Jakobsweg | German |
| IT | Cammino di Santiago | Italian |
| US | Camino de Santiago | English-US |
| GB | Camino de Santiago | English-UK |
| PT | Caminho de Santiago | Portuguese |
| KR | 산티아고 순례길 | Korean |
| BR | Caminho de Santiago | Portuguese-BR |

In [ ]:
# LOAD
trends_annual = pd.read_csv(PROCESSED / 'trends_annual.csv')
lang_map = dict(zip(trends_annual['geo'], trends_annual['language']))
print('trends_annual: ' + str(trends_annual.shape))
print('Countries: ' + str(sorted(trends_annual['geo'].unique())))
print('Years: ' + str(trends_annual['year'].min()) + ' to ' + str(trends_annual['year'].max()))

In [ ]:
# COLLECTION — re-run only to refresh Trends data
# Runtime: ~3 min | Requires: pip install pytrends
# Note: TrendReq must be initialised without retries/backoff_factor (urllib3 >= 2.0 compatibility)

from pytrends.request import TrendReq

KEYWORDS = {
    'ES': ('Camino de Santiago',    'Spanish'),
    'FR': ('chemin de Compostelle', 'French'),
    'DE': ('Jakobsweg',             'German'),
    'IT': ('Cammino di Santiago',   'Italian'),
    'US': ('Camino de Santiago',    'English-US'),
    'GB': ('Camino de Santiago',    'English-UK'),
    'PT': ('Caminho de Santiago',   'Portuguese'),
    'KR': ('산티아고 순례길',           'Korean'),
    'BR': ('Caminho de Santiago',   'Portuguese-BR'),
}

def fetch_trends_country(pt, keyword, geo):
    try:
        pt.build_payload([keyword], geo=geo, timeframe='2004-01-01 2024-12-31')
        df = pt.interest_over_time()
        if df.empty:
            return None
        df = df.reset_index()[['date', keyword]]
        df.columns = ['date', 'interest']
        df['keyword'] = keyword
        df['geo'] = geo
        return df
    except Exception as e:
        print('  error ' + geo + ': ' + str(e))
        return None

pt = TrendReq(hl='en-US', tz=0)
all_trends, failed = [], []

for geo, (keyword, lang) in KEYWORDS.items():
    print('Fetching ' + geo + ' ...', end=' ')
    df = fetch_trends_country(pt, keyword, geo)
    if df is not None:
        all_trends.append(df)
        print('ok ' + str(len(df)) + ' rows')
    else:
        failed.append(geo)
    time.sleep(2)

# Retry failed (HTTP 429 rate limiting)
for geo in failed[:]:
    keyword, _ = KEYWORDS[geo]
    time.sleep(5)
    df = fetch_trends_country(pt, keyword, geo)
    if df is not None:
        all_trends.append(df)
        failed.remove(geo)
        print('retry ok: ' + geo)

trends_raw = pd.concat(all_trends, ignore_index=True)
trends_raw['date'] = pd.to_datetime(trends_raw['date'])
trends_raw['year'] = trends_raw['date'].dt.year
trends_raw.to_csv(RAW / 'trends_raw_2004_2024.csv', index=False)

lang_map = {geo: lang for geo, (_, lang) in KEYWORDS.items()}
trends_annual = (
    trends_raw.groupby(['geo','year'])['interest']
    .mean().round(2).reset_index()
    .rename(columns={'interest':'interest_mean'})
)
trends_annual['language'] = trends_annual['geo'].map(lang_map)
trends_annual.to_csv(PROCESSED / 'trends_annual.csv', index=False)
print('Done. Failed: ' + str(failed))

In [ ]:
# Visualisation
countries = sorted(trends_annual['geo'].unique())
palette   = sns.color_palette('muted', len(countries))
events    = {2010:'Holy Year', 2015:'Holy Year', 2020:'COVID', 2021:'Holy Year'}

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('Google Trends — Camino de Santiago interest by country (2004-2024)', fontsize=13)

ax = axes[0]
for i, geo in enumerate(countries):
    s = trends_annual[trends_annual['geo'] == geo]
    ax.plot(s['year'], s['interest_mean'], marker='o', markersize=4, linewidth=1.8,
            label=geo + ' - ' + str(lang_map.get(geo,'')), color=palette[i])
for yr, label in events.items():
    ax.axvline(x=yr, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.text(yr+0.1, ax.get_ylim()[1]*0.95, label, fontsize=8, color='gray', rotation=90, va='top')
ax.set_title('Annual mean interest (0-100, relative)')
ax.legend(fontsize=8, loc='upper left')

pivot = trends_annual.pivot(index='geo', columns='year', values='interest_mean')
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, ax=axes[1],
            cbar_kws={'label':'Mean interest (0-100)'})
axes[1].set_title('Heatmap — country x year')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'trends_annual_by_country.png', dpi=150)
plt.show()

### Key observations — Google Trends 2004–2024

**General trend**: Structural growth across all countries, accelerating from 2012–2013. Global visibility has grown continuously over two decades.

**US — isolated spike in 2012**: Strong peak ahead of the general trend. Likely linked to wide distribution of the film *The Way* (2010–2011). One-year lag typical of deferred notoriety effects. Cross-check against Oficina US pilgrim counts 2012–2013. Strong Axis B candidate signal.

**DE — isolated peak in 2007**: Before all other countries. Likely a major German book or TV documentary. Requires investigation for Axis B media event database.

**KR — near-zero before 2011, then steady linear growth**: No single triggering event — gradual cultural diffusion. Consistent with Oficina data showing Korean pilgrim explosion from 2012 onward.

**COVID 2020**: Dip across all 9 countries — confirms data consistency.

**FR — accelerating post-2021**: Highest recorded values in 2023–2024. Strong Axis C signal.

**PT — double peak**: 2019 peak aligns with Camino Portugues explosion. Second peak in 2024.

---
## Section 2 — Oficina del Peregrino

**Objective**: Extract annual pilgrim statistics from official Oficina del Peregrino publications (2004–2024).

**Data coverage**:
- 2004–2018: PDF extraction via `pdfplumber` (annual reports downloaded manually from oficinadelperegrino.com)
- 2019–2024: Manual entry from official publications (Oficina web interface + americanpilgrims.org)

**Known structural bias**: Statistics count only pilgrims who collected a Compostela in Santiago. Xunta de Galice IoT sensor study (2024–25) estimates actual pilgrim numbers are ~40% higher. Consistent across years — trend analysis valid, absolute counts underestimated.

In [ ]:
# LOAD
df_master      = pd.read_csv(PROCESSED / 'master_annual.csv')
df_routes_full = pd.read_csv(PROCESSED / 'master_routes.csv')
df_nat         = pd.read_csv(PROCESSED / 'oficina_nationalities.csv')
df_prof        = pd.read_csv(PROCESSED / 'oficina_professions.csv')

print('master_annual        : ' + str(df_master.shape))
print('master_routes        : ' + str(df_routes_full.shape))
print('oficina_nationalities: ' + str(df_nat.shape))
print('oficina_professions  : ' + str(df_prof.shape))

In [ ]:
# COLLECTION — PDF extraction 2004–2018
# Requires: pip install pdfplumber
# Requires: PDF files in data/raw/oficina/ named YYYY_oficina.pdf

import pdfplumber

def extract_total(text):
    m = re.search(r'total absoluto de ([\d\.]+)', text)
    return int(m.group(1).replace('.','')) if m else None

def clean_count(v):
    m = re.search(r'([\d\.]+)', str(v))
    return int(m.group(1).replace('.','')) if m else None

def extract_oficina_pdf(pdf_path, year):
    result = {'year': year}
    with pdfplumber.open(pdf_path) as pdf:
        text2   = pdf.pages[1].extract_text() or ''
        tables2 = pdf.pages[1].extract_tables()
        result['total_pilgrims'] = extract_total(text2)
        for t in tables2:
            if not t or len(t) < 2: continue
            h = t[0]
            if h[0] == 'Sexo':
                for row in t[1:]:
                    if row[0] and 'Mujer'  in str(row[0]): result['women'] = clean_count(row[1])
                    if row[0] and 'Hombre' in str(row[0]): result['men']   = clean_count(row[1])
            elif h[0] == 'Medio':
                for row in t[1:]:
                    if row[0]: result['means_'+row[0].lower().replace(' ','_')] = clean_count(row[1])
            elif h[0] == 'Motivos':
                for row in t[1:]:
                    if row[0]: result['motive_'+row[0].lower()[:12].replace(' ','_')] = clean_count(row[1])
            elif h[0] == 'Edad':
                for row in t[1:]:
                    if row[0]: result['age_'+row[0].replace(' ','').replace('<','u').replace('>','o')] = clean_count(row[1])
        nats = {}
        for page in pdf.pages[2:5]:
            for t in page.extract_tables():
                if t and t[0][0] in ('Pais','País'):
                    for row in t[1:]:
                        if row[0] and row[1]: nats[row[0].strip()] = clean_count(row[1])
        result['nationalities'] = nats
        routes = {}
        for page in pdf.pages:
            for t in page.extract_tables():
                if t and t[0][0] == 'Camino':
                    for row in t[1:]:
                        if row[0] and row[1]: routes[row[0].strip()] = clean_count(row[1])
        result['routes'] = routes
        profs = {}
        for page in pdf.pages:
            for t in page.extract_tables():
                if t and t[0][0] == 'Profesiones':
                    for row in t[1:]:
                        if row[0] and row[1]: profs[row[0].strip()] = clean_count(row[1])
        result['professions'] = profs
    return result

all_years_data = []
for pdf_path in sorted((RAW / 'oficina').glob('*.pdf')):
    year = int(pdf_path.stem.split('_')[0])
    print('Extracting ' + str(year) + ' ...', end=' ')
    data = extract_oficina_pdf(pdf_path, year)
    all_years_data.append(data)
    print('ok total=' + str(data.get('total_pilgrims')))
print('Extracted: ' + str(len(all_years_data)) + ' years')

In [ ]:
# COLLECTION — Build master_annual + master_routes
# Run after PDF extraction cell above

# Build DataFrames from PDF data
summary_rows, route_rows, nat_rows, prof_rows = [], [], [], []
for d in all_years_data:
    summary_rows.append({k:v for k,v in d.items() if k not in ('nationalities','routes','professions')})
    for route, count in d.get('routes',{}).items():
        route_rows.append({'year':d['year'],'route':route,'count':count})
    for country, count in d.get('nationalities',{}).items():
        nat_rows.append({'year':d['year'],'country':country,'count':count})
    for prof, count in d.get('professions',{}).items():
        prof_rows.append({'year':d['year'],'profession':prof,'count':count})

df_pdf = pd.DataFrame(summary_rows).sort_values('year').reset_index(drop=True)

# Merge duplicate columns (spelling variants across years)
for new, old in [('means_bicicleta','means_biciclet'),
                 ('motive_religious','motive_religioso'),
                 ('motive_religious_mix','motive_religioso_y_'),
                 ('motive_non_religious','motive_no_religioso')]:
    if old in df_pdf.columns:
        df_pdf[new] = df_pdf.get(new, pd.Series(dtype=float)).combine_first(df_pdf[old])
        df_pdf = df_pdf.drop(columns=[old])
for new, old in [('motive_religious','motive_religiosa'),
                 ('motive_religious_mix','motive_religiosa_y_'),
                 ('motive_non_religious','motive_no_religiosa')]:
    if old in df_pdf.columns:
        df_pdf[new] = df_pdf.get(new, pd.Series(dtype=float)).combine_first(df_pdf[old])
        df_pdf = df_pdf.drop(columns=[old])

df_pdf = df_pdf.rename(columns={
    'means_pie':'means_foot','means_bicicleta':'means_bike',
    'means_caballo':'means_horse','means_silla_de_ruedas':'means_wheelchair',
    'means_vela':'means_sailboat','age_u30':'age_under30',
    'age_30-60':'age_30to60','age_o60':'age_over60'})
df_pdf['pct_women'] = (df_pdf['women'] / df_pdf['total_pilgrims'] * 100).round(2)

# Manual data 2019–2024
manual_data = [
    {'year':2019,'total_pilgrims':347566,'women':181576,'men':165990,'means_foot':322640,'means_bike':23560,'motive_religious':140898,'motive_religious_mix':163129,'motive_non_religious':43539},
    {'year':2020,'total_pilgrims': 53983,'women': 28044,'men': 25939,'means_foot': 49857,'means_bike': 3817,'motive_religious': 21099,'motive_religious_mix': 24745,'motive_non_religious': 8139},
    {'year':2021,'total_pilgrims':175458,'women': 92350,'men': 83108,'means_foot':162891,'means_bike':11987,'motive_religious': 64419,'motive_religious_mix': 84731,'motive_non_religious':26308},
    {'year':2022,'total_pilgrims':438209,'women':232050,'men':206159,'means_foot':407769,'means_bike':29108,'motive_religious':177417,'motive_religious_mix':209314,'motive_non_religious':51478},
    {'year':2023,'total_pilgrims':446051,'women':237143,'men':208908,'means_foot':415647,'means_bike':29282,'motive_religious':179876,'motive_religious_mix':210687,'motive_non_religious':55488},
    {'year':2024,'total_pilgrims':499186,'women':264968,'men':234218,'means_foot':464644,'means_bike':33218,'motive_religious':198762,'motive_religious_mix':241309,'motive_non_religious':59115},
]
df_manual = pd.DataFrame(manual_data)
df_manual['pct_women'] = (df_manual['women'] / df_manual['total_pilgrims'] * 100).round(2)

cols = ['year','total_pilgrims','women','men','pct_women','means_foot','means_bike',
        'motive_religious','motive_religious_mix','motive_non_religious']
df_master = pd.concat([df_pdf[cols], df_manual[cols]], ignore_index=True).sort_values('year').reset_index(drop=True)
df_master['holy_year']         = df_master['year'].isin([2004,2010,2021]).astype(int)
df_master['covid_year']        = df_master['year'].isin([2020,2021]).astype(int)
df_master['pct_foot']          = (df_master['means_foot']/df_master['total_pilgrims']*100).round(2)
df_master['pct_bike']          = (df_master['means_bike']/df_master['total_pilgrims']*100).round(2)
df_master['pct_religious']     = (df_master['motive_religious']/df_master['total_pilgrims']*100).round(2)
df_master['pct_non_religious'] = (df_master['motive_non_religious']/df_master['total_pilgrims']*100).round(2)
df_master['yoy_growth']        = df_master['total_pilgrims'].pct_change().round(4)
df_master.to_csv(PROCESSED / 'master_annual.csv', index=False)

# Routes — clean PDF artefacts + append manual 2019–2024
df_routes_pdf = pd.DataFrame(route_rows)
route_clean = {
    'Frances-Camino de':          'Camino Frances',
    'Portugues-Camino':           'Camino Portugues',
    'Norte-Camino de':            'Camino del Norte',
    'Primitivo-Camino':           'Camino Primitivo',
    'Ingles-Camino':              'Camino Ingles',
    'Via de la Plata':            'Via de la Plata',
    'Camino Portugues de la costa':'Camino Portugues Costa',
    'Camino Frances':             'Camino Frances',
    'Camino Portugues':           'Camino Portugues',
}
df_routes_pdf['route'] = df_routes_pdf['route'].replace(route_clean)
df_routes_pdf = df_routes_pdf.groupby(['year','route'], as_index=False)['count'].sum()
df_routes_pdf.to_csv(PROCESSED / 'master_routes.csv', index=False)

pd.DataFrame(nat_rows).sort_values(['year','country']).to_csv(PROCESSED / 'oficina_nationalities.csv', index=False)
pd.DataFrame(prof_rows).sort_values(['year','profession']).to_csv(PROCESSED / 'oficina_professions.csv', index=False)
print('All files saved.')

In [ ]:
# Visualisation — master annual indicators
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Camino de Santiago — Master indicators 2004-2024', fontsize=14)
years = df_master['year']

ax = axes[0,0]
colors_bar = ['#c0392b' if hy else '#2980b9' for hy in df_master['holy_year']]
ax.bar(years, df_master['total_pilgrims']/1000, color=colors_bar, alpha=0.85)
ax.axvspan(2019.5, 2021.5, alpha=0.1, color='gray', label='COVID period')
ax.set_title('Total pilgrims (thousands) - Red = Holy Year')
ax.legend(fontsize=8)

ax = axes[0,1]
ax.plot(years, df_master['pct_women'], marker='o', color='#e91e63', linewidth=2)
ax.axhline(y=50, color='gray', linestyle='--', linewidth=0.8)
ax.fill_between(years, 50, df_master['pct_women'],
                where=df_master['pct_women']>=50, alpha=0.15, color='#e91e63')
ax.set_title('% Women pilgrims')
ax.set_ylim(38, 56)

ax = axes[1,0]
ax.plot(years, df_master['pct_foot'], marker='o', color='#2196F3', linewidth=2, label='Foot %')
ax.plot(years, df_master['pct_bike'], marker='s', color='#FF9800', linewidth=2, label='Bike %')
ax.set_title('Transport mode share (%)')
ax.legend(fontsize=9)

ax = axes[1,1]
ax.stackplot(years,
    df_master['pct_religious'],
    df_master['motive_religious_mix']/df_master['total_pilgrims']*100,
    df_master['pct_non_religious'],
    labels=['Religious only','Religious + other','Non-religious'],
    colors=['#c0392b','#e67e22','#27ae60'], alpha=0.8)
ax.set_title('Declared motives (%)')
ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'master_annual.png', dpi=150)
plt.show()

### Key observations — Master annual 2004–2024

**Total pilgrims**: Near-linear growth from 179,944 (2004) to 499,186 (2024), x3 over 20 years. COVID 2020 is the only true interruption (-84%). 2024 exceeds 2019 pre-COVID level by 44%. Holy Year effect quantified: 2010 +86.5% vs 2009, 2021 +125% vs 2020.

**% Women — two-phase feminisation**: Phase 1 (2004–2012): unstable plateau at 40–43%. Phase 2 (2013–2024): continuous acceleration to 53%. Parity crossed in 2018, never reversed. Strong Axis D signal.

**Transport**: pct_foot 82% to 93%, pct_bike 18% to 7%. The new pilgrim is overwhelmingly a walker. Bike peaks at Holy Years (atypical profile years).

**Motives**: Non-religious share grows from ~5% (2004) to ~15% (2024). Long-term secularisation trend is real but gradual. Caution: self-declared data, Compostela-eligibility bias applies.

In [ ]:
# Visualisation — routes full series 2004–2024
route_colors = {
    'Camino Frances':        '#2196F3',
    'Camino Portugues':      '#E91E63',
    'Camino del Norte':      '#4CAF50',
    'Via de la Plata':       '#FF9800',
    'Camino Primitivo':      '#9C27B0',
    'Camino Ingles':         '#00BCD4',
    'Camino Portugues Costa':'#F44336',
    'Camino de Invierno':    '#795548',
    'Muxia-Finisterre':      '#607D8B',
    'Other routes':          '#9E9E9E',
}

routes_pivot = df_routes_full.pivot_table(
    index='year', columns='route', values='count', aggfunc='sum').fillna(0)
years_full = routes_pivot.index.astype(int).tolist()

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle('Camino routes — Full series 2004-2024', fontsize=14)

ax = axes[0]
for route in routes_pivot.columns:
    if routes_pivot[route].sum() > 0:
        lw = 2.5 if route in ['Camino Frances','Camino Portugues','Camino Portugues Costa'] else 1.5
        ax.plot(years_full, routes_pivot[route]/1000, marker='o', markersize=3,
                linewidth=lw, label=route, color=route_colors.get(route,'#999'))
ax.axvspan(2019.5, 2021.5, alpha=0.08, color='gray')
ax.set_title('Absolute flows (thousands)')
ax.legend(fontsize=8, loc='upper left')

ax = axes[1]
bottom = np.zeros(len(years_full))
for route in routes_pivot.columns:
    if routes_pivot[route].sum() > 0:
        values = routes_pivot[route].values / routes_pivot.sum(axis=1).values * 100
        ax.bar(years_full, values, bottom=bottom, label=route,
               color=route_colors.get(route,'#999'), alpha=0.85)
        bottom += values
ax.set_title('% share by route')
ax.set_xticks(years_full)
ax.set_xticklabels(years_full, rotation=45)
ax.legend(fontsize=8, loc='lower left')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'master_routes.png', dpi=150)
plt.show()

### Key observations — Routes 2004–2024

**Camino Frances**: Share drops from ~77% (2004) to ~45% (2024). Absolute numbers grow but pace overtaken by other routes. Core Axis C signal.

**Camino Portugues Costa — standout signal**: Near-zero before 2018, ~90k in 2024. x18 growth from 2020 to 2024. By 2024 nearly matches Camino Portugues central. Clearest diversification signal in the dataset.

**Combined Portuguese corridor**: ~38% of all pilgrims in 2024 — nearly as much as the Frances alone.

**Hypotheses for Portugues Costa explosive growth**:
- H1 (Low-cost aviation): expansion of low-cost routes to Porto from European capitals lowered access cost. Testable with OpenFlights/OAG data.
- H2 (Primo-pilgrim profile): growth may reflect first-time pilgrims bypassing the Frances entirely, attracted by shorter distance and coastal scenery.

**Known data quality note**: Small rounding discrepancies in route totals vs official totals for 2010 (-1,000), 2013 (-1), 2014 (-3), 2015 (-14), 2018 (-1). PDF extraction artefacts, negligible relative to volumes.

---
## Section 3 — Trail du Saint-Jacques by UTMB

**Objective**: Collect finisher data (2012–2024) to build a sport-spillover proxy for Axis B.

**Source**: LiveTrail XML API at `tsj.livetrail.net`

**Context**: Since 2012 the trail positions itself as a *spiritrail* — a cultural and spiritual journey on and around the Camino route. Integration into the UTMB World Series (~2022) drove explosive international growth.

In [ ]:
# LOAD
trail_annual = pd.read_csv(PROCESSED / 'trail_sj_annual.csv')
trail_nat    = pd.read_csv(PROCESSED / 'trail_sj_nationalities.csv')
print('trail_annual : ' + str(trail_annual.shape))
print('trail_nat    : ' + str(trail_nat.shape))
print('Years: ' + str(sorted(trail_annual['year'].unique())))

In [ ]:
# COLLECTION — LiveTrail XML scraping
# Runtime: ~5 min | Requires: requests
# Note: runner data requires ?course=RACE_ID&cat=scratch&pays=all parameter combination

import requests

EDITIONS = {
    2012:'http://livetrail.net/histo/gtsj2012',
    2013:'http://livetrail.net/histo/gtsj_2013',
    2014:'http://livetrail.net/histo/gtsj_2014',
    2015:'http://livetrail.net/histo/gtsj_2015',
    2016:'http://livetrail.net/histo/gtsj_2016',
    2017:'http://livetrail.net/histo/gtsj_2017',
    2018:'http://livetrail.net/histo/gtsj_2018',
    2019:'http://livetrail.net/histo/gtsj_2019',
    2021:'http://livetrail.net/histo/gtsj_2021',
    2022:'http://livetrail.net/histo/tsj_2022',
    2023:'http://livetrail.net/histo/tsj_2023',
    2024:'http://livetrail.net/histo/tsj_2024',
}
HEADERS = {'User-Agent': 'Mozilla/5.0'}

def fetch_edition_runners(base_url, year):
    runners = []
    root = ET.fromstring(requests.get(base_url+'/classement.php', headers=HEADERS, timeout=10).content)
    races = [c.get('id') for c in root.findall('.//courses/c')]
    for race_id in races:
        url = base_url + '/classement.php?course=' + race_id + '&cat=scratch&pays=all'
        try:
            root2 = ET.fromstring(requests.get(url, headers=HEADERS, timeout=15).content)
            for r in root2.findall('.//classement//c'):
                runners.append({'year':year,'race':race_id,
                    'country':r.get('cod',r.get('p','')),'country_name':r.get('pays',''),
                    'sex':r.get('sx',''),'cat':r.get('cat',''),'rank':r.get('class',''),
                    'time':r.get('tps','')})
            time.sleep(0.5)
        except Exception as e:
            print('  ' + race_id + ': ' + str(e))
    return runners

all_runners = []
for year, base_url in EDITIONS.items():
    print('Fetching ' + str(year) + ' ...', end=' ')
    try:
        runners = fetch_edition_runners(base_url, year)
        all_runners.extend(runners)
        print('ok ' + str(len(runners)) + ' runners')
    except Exception as e:
        print('error: ' + str(e))
    time.sleep(1)

df_trail = pd.DataFrame(all_runners)
df_trail.to_csv(RAW / 'trail_sj_runners_raw.csv', index=False)

trail_annual = df_trail.groupby('year').agg(
    total_finishers=('rank','count'),
    n_countries=('country','nunique'),
    pct_women=('sex', lambda x: round((x=='F').mean()*100, 2))
).reset_index()

trail_nat = (df_trail.groupby(['year','country','country_name'])
    .size().reset_index(name='finishers')
    .sort_values(['year','finishers'], ascending=[True,False]))

trail_annual.to_csv(PROCESSED / 'trail_sj_annual.csv', index=False)
trail_nat.to_csv(PROCESSED / 'trail_sj_nationalities.csv', index=False)
print('Saved. Total: ' + str(len(all_runners)) + ' runners')

In [ ]:
# Visualisation
known_issue = [2014,2015,2016,2017]
years_t     = trail_annual['year'].tolist()
nat_counts  = trail_nat.groupby('year')['country'].nunique().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Trail du Saint-Jacques by UTMB — Key indicators 2013-2024', fontsize=13)

ax = axes[0]
ax.bar(years_t, trail_annual['total_finishers'], color='#9C27B0', alpha=0.85)
ax.axvspan(2019.5, 2021.5, alpha=0.08, color='gray')
ax.set_title('Total finishers (all races)')
ax.set_xticks(years_t)
ax.set_xticklabels(years_t, rotation=45)

ax = axes[1]
colors_nat = ['#FF9800' if y in known_issue else '#9C27B0' for y in nat_counts['year']]
ax.bar(nat_counts['year'], nat_counts['country'], color=colors_nat, alpha=0.85)
ax.set_title('Countries (orange = data incomplete)')
ax.set_xticks(nat_counts['year'].tolist())
ax.set_xticklabels(nat_counts['year'].tolist(), rotation=45)

ax = axes[2]
ax.plot(trail_annual['year'], trail_annual['pct_women'], marker='o', color='#E91E63', linewidth=2)
ax.axhline(y=50, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('% Women finishers')
ax.set_ylim(0, 55)
ax.set_xticks(years_t)
ax.set_xticklabels(years_t, rotation=45)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'trail_sj_annual.png', dpi=150)
plt.show()

### Key observations — Trail du Saint-Jacques 2013–2024

**Two growth phases**: Stable plateau ~1,300–1,800 finishers (2013–2019), then explosive growth x2.1 in two years (2022–2024). Break point coincides with UTMB World Series integration.

**Internationalisation**: Countries jump from 7–12 (2018–2021) to 27–49 (2022–2024). UTMB label brings international audience the event could not attract independently. 49 countries exposed to Camino territory in 2024. Top nationalities: France (3,870), Belgium (67), UK (44), Spain (35), Netherlands (31).

**% Women ~20–30%**: Well below Camino pilgrim share (53%). Trail and pilgrimage communities are culturally distinct. Any spillover would skew toward male profiles.

**Axis B integration plan**: Trail finisher count and international diversity index used as proxy features in NB03. Hypothesis: trail growth years precede pilgrim increases from trail-active countries (FR, BE, GB, NL) with 0–2 year lag.

**Known data quality issues**:
- 2012: XML malformed — excluded. Coverage: 11/12 editions (no 2020 COVID).
- 2014–2017: country field absent — nationality analysis restricted to 2018–2024.

---
## Section 4 — ERA5 Climate data

**Objective**: Monthly temperature and precipitation for 5 route corridors (2004–2024).

**Source**: Copernicus CDS — ERA5 monthly averaged reanalysis (`reanalysis-era5-single-levels-monthly-means`).

**Route corridors** (bounding boxes [N, W, S, E]):

| Route | BBox |
|-------|------|
| camino_frances | [44.0, -8.5, 42.0, -1.0] |
| camino_portugues | [43.0, -8.8, 38.5, -7.5] |
| camino_portugues_costa | [43.0, -9.0, 40.5, -8.0] |
| camino_norte | [44.0, -8.5, 43.0, -1.5] |
| via_podiensis_fr | [46.0, -4.0, 43.0, 1.5] |

**Known limitation**: Spatial average over bbox — does not capture within-corridor microclimates. Sufficient for annual signal detection.

In [ ]:
# LOAD
df_climate = pd.read_csv(PROCESSED / 'era5_climate_monthly.csv')
print('era5_climate_monthly: ' + str(df_climate.shape))
print('Routes: ' + str(sorted(df_climate['route'].unique())))
print('Years: ' + str(df_climate['year'].min()) + ' to ' + str(df_climate['year'].max()))

In [ ]:
# COLLECTION — ERA5 download via CDS API
# Runtime: ~30 min for 105 requests (21 years x 5 routes)
# Requires: cdsapi, xarray, ~/.cdsapirc configured
# Note: CDS returns ZIP archive — extract to unique temp dir per request to avoid Windows file locks

import cdsapi
import xarray as xr

ROUTE_CORRIDORS = {
    'camino_frances':        {'bbox': [44.0, -8.5, 42.0, -1.0]},
    'camino_portugues':      {'bbox': [43.0, -8.8, 38.5, -7.5]},
    'camino_portugues_costa':{'bbox': [43.0, -9.0, 40.5, -8.0]},
    'camino_norte':          {'bbox': [44.0, -8.5, 43.0, -1.5]},
    'via_podiensis_fr':      {'bbox': [46.0, -4.0, 43.0,  1.5]},
}

def extract_corridor_means(year, route_name, bbox):
    zip_path = EXTERNAL / ('era5_' + str(year) + '_' + route_name + '.zip')
    temp_dir = EXTERNAL / ('tmp_' + str(year) + '_' + route_name)
    temp_dir.mkdir(exist_ok=True)
    cdsapi.Client().retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {'product_type':['monthly_averaged_reanalysis'],
         'variable':['2m_temperature','total_precipitation'],
         'year':[str(year)],'month':['01','02','03','04','05','06','07','08','09','10','11','12'],
         'time':['00:00'],'area':bbox,'data_format':'netcdf','download_format':'unarchived'},
        str(zip_path))
    nc_temp = nc_prec = None
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(temp_dir)
        for name in z.namelist():
            if 'avgua' in name: nc_temp = temp_dir / name
            elif 'avgad' in name: nc_prec = temp_dir / name
    rows = []
    ds_t = xr.open_dataset(nc_temp)
    ds_p = xr.open_dataset(nc_prec)
    for m in range(12):
        rows.append({
            'year':year,'month':m+1,'route':route_name,
            'temp_c':   round(float(ds_t['t2m'].isel(valid_time=m).mean().values)-273.15, 2),
            'precip_mm':round(float(ds_p['tp'].isel(valid_time=m).mean().values)*1000*30, 2),
        })
    ds_t.close()
    ds_p.close()
    zip_path.unlink(missing_ok=True)
    shutil.rmtree(temp_dir)
    return pd.DataFrame(rows)

all_climate, failed_requests = [], []
total = len(range(2004,2025)) * len(ROUTE_CORRIDORS)
done  = 0

for year in range(2004, 2025):
    for route_name, info in ROUTE_CORRIDORS.items():
        done += 1
        print(str(done)+'/'+str(total)+' - '+str(year)+' '+route_name+' ...', end=' ')
        try:
            all_climate.append(extract_corridor_means(year, route_name, info['bbox']))
            print('ok')
        except Exception as e:
            print('error: ' + str(e)[:80])
            failed_requests.append((year, route_name))
        time.sleep(2)

df_climate = pd.concat(all_climate, ignore_index=True)
df_climate.to_csv(PROCESSED / 'era5_climate_monthly.csv', index=False)
print('Done! Shape: ' + str(df_climate.shape) + ' | Failed: ' + str(len(failed_requests)))

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('ERA5 Climate — Monthly means by route corridor 2004-2024', fontsize=13)

for ax, col, title, ylabel in [
    (axes[0], 'temp_c',    'Mean annual temperature (C)',     'C'),
    (axes[1], 'precip_mm', 'Mean monthly precipitation (mm)', 'mm/month')
]:
    data = df_climate.groupby(['year','route'])[col].mean().reset_index()
    for route in data['route'].unique():
        s = data[data['route']==route]
        ax.plot(s['year'], s[col], marker='o', markersize=3, linewidth=1.5, label=route)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'era5_climate_overview.png', dpi=150)
plt.show()

### Key observations — ERA5 Climate 2004–2024

**Temperature hierarchy consistent with geography**: Camino Portugues warmest (~14–15°C), Camino Frances coolest (~12–13°C). Slight warming trend post-2020 — relevant for Axis A heat stress modelling.

**Precipitation — Portugues Costa most variable**: Range 60–155 mm/month, Atlantic exposure. Via Podiensis FR lowest overall.

**Axis A hypothesis now testable**: Whether anomalously hot summers (Frances, Norte) or wet springs (Portugues Costa) correlate with flow deflection toward alternative routes in the same year or with a 1-year lag.

---
## Summary — Processed datasets

| File | Shape | Content |
|------|-------|---------|
| `trends_annual.csv` | (189, 4) | Google Trends annual index, 9 countries, 2004–2024 |
| `master_annual.csv` | (21, 17) | Official annual totals + derived features, 2004–2024 |
| `master_routes.csv` | (180, 3) | Pilgrim counts by route and year, 2004–2024 |
| `oficina_nationalities.csv` | (1974, 3) | Nationalities from PDFs, 2004–2018 |
| `oficina_professions.csv` | (261, 3) | Professions from PDFs, 2004–2018 |
| `trail_sj_annual.csv` | (11, 4) | Trail finishers, countries, % women by year |
| `trail_sj_nationalities.csv` | — | Trail finisher counts by country and year |
| `era5_climate_monthly.csv` | (1260, 5) | Monthly temp + precip, 5 corridors, 2004–2024 |

**Next step**: NB02 — Exploratory data analysis, structural break detection, axis validation.

---
## Development notes

Key difficulties encountered during collection — relevant for NB05 limitations section.

**Section 1 — Google Trends**:
- `pytrends` v4.9.2 incompatible with `urllib3 >= 2.0` (`method_whitelist` argument removed). Fix: initialise `TrendReq` without `retries` and `backoff_factor`.
- HTTP 429 rate limiting occurs randomly. Fix: 2s pause between requests + retry with 5s pause.

**Section 2 — Oficina del Peregrino**:
- PDFs only downloadable up to 2018. Years 2019–2024 entered manually from official publications.
- Routes table absent from PDFs before 2015 (8-page format). Fix: search all pages for header == 'Camino'.
- PDFs 2014–2015: two-column route table parsed in wrong order (artefact names: Frances-Camino de, etc.). Fix: explicit renaming dict applied post-extraction.
- Via de la Plata / Via de la Plata spelling variant across years. Fix: merged in renaming dict.
- Oficina website 2019+: data loaded via JavaScript/WordPress AJAX, not accessible with `requests`. WordPress action `compostelana_get_estadisticas` identified but returns empty array (requires session nonce). Manual entry used as workaround.

**Section 3 — Trail du Saint-Jacques**:
- 2012 edition: XML malformed. Excluded.
- 2014–2017: country code field absent. Nationality data unreliable for these years.
- Runner data requires `?course=RACE_ID&cat=scratch&pays=all` parameter combination.

**Section 4 — ERA5**:
- New CDS API (2024): dataset renamed, parameters use lists, `format` split into `data_format` + `download_format`.
- CDS returns ZIP despite `download_format: unarchived`. Fix: extract to unique temp dir per request.
- Windows file locking: `.nc` files cannot be deleted while open in xarray. Fix: explicit `ds.close()` before `shutil.rmtree(temp_dir)`.
- API key stored in `~/.cdsapirc` — never committed to Git.